# EcoHome Energy Advisor - RAG Setup

In this notebook, you'll set up the Retrieval-Augmented Generation (RAG) pipeline for the EcoHome Energy Advisor. This will allow the agent to access and cite relevant energy-saving tips and best practices.

## Learning Objectives
- Set up ChromaDB vector store
- Load and process energy-saving documents
- Create embeddings for document chunks
- Implement semantic search functionality
- Test the RAG pipeline

## Documents Available
Starter documents:
- `tip_device_best_practices.txt` - Device-specific optimization tips
- `tip_energy_savings.txt` - General energy-saving strategies

Added to expand the knowledge base:
- `tip_hvac_optimization.txt` - Thermostat setpoints, pre-cooling around peak prices, heat pumps, maintenance
- `tip_smart_home_automation.txt` - Smart thermostats, price/solar-aware automations, smart plugs, monitoring
- `tip_renewable_energy_integration.txt` - Solar self-consumption, using weather forecasts, net metering
- `tip_seasonal_energy_management.txt` - Summer, fall, winter and spring strategies
- `tip_energy_storage_optimization.txt` - Home battery modes, scheduling, EVs and water heaters as storage, ROI
- `tip_ev_charging.txt` - EV energy use, best charging times, battery health, cost examples
- `tip_time_of_use_pricing.txt` - TOU periods, load shifting, savings calculations

## 1. Import Required Libraries


In [1]:
# Import the necessary libraries for RAG setup
import os
from langchain_chroma  import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

## 2. Load and Process Documents


In [3]:
# Load the energy-saving tip documents
# Load every .txt file in data/documents (the 2 starter documents plus the added ones)
# Use TextLoader to load the documents
import glob

documents = []
document_paths = sorted(glob.glob("data/documents/*.txt"))

for doc_path in document_paths:
    loader = TextLoader(doc_path, encoding="utf-8")
    docs = loader.load()
    documents.extend(docs)
    print(f"Loaded {len(docs)} documents from {doc_path}")

print(f"Total documents loaded: {len(documents)}")

Loaded 1 documents from data/documents/tip_device_best_practices.txt
Loaded 1 documents from data/documents/tip_energy_savings.txt
Loaded 1 documents from data/documents/tip_energy_storage_optimization.txt
Loaded 1 documents from data/documents/tip_ev_charging.txt
Loaded 1 documents from data/documents/tip_hvac_optimization.txt
Loaded 1 documents from data/documents/tip_renewable_energy_integration.txt
Loaded 1 documents from data/documents/tip_seasonal_energy_management.txt
Loaded 1 documents from data/documents/tip_smart_home_automation.txt
Loaded 1 documents from data/documents/tip_time_of_use_pricing.txt
Total documents loaded: 9


## 3. Split Documents into Chunks


In [4]:
# Split documents into smaller chunks for better retrieval
# Use RecursiveCharacterTextSplitter with appropriate chunk_size and chunk_overlap
# Experiment with different chunk sizes (e.g., 500, 1000, 1500 characters)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)

# Split the documents
splits = text_splitter.split_documents(documents)
print(f"Split {len(documents)} documents into {len(splits)} chunks")

# Show sample chunk
if splits:
    print(f"\nSample chunk (first 200 characters):")
    print(splits[0].page_content[:200] + "...")


Split 9 documents into 27 chunks

Sample chunk (first 200 characters):
Large devices like electric vehicles, washing machines and dishwashers often support delayed start or timer functions. Schedule these devices to run outside of peak electricity pricing hours or during...


## 4. Create Vector Store


In [5]:
# Create a ChromaDB vector store
# Initialize OpenAIEmbeddings
# Create the vector store with the document chunks
# Persist the vector store to disk for future use
import shutil

# Set up the persist directory.
# Remove any previous store first so re-running this notebook doesn't index the same chunks twice.
persist_directory = "data/vectorstore"
if os.path.exists(persist_directory):
    shutil.rmtree(persist_directory)
os.makedirs(persist_directory, exist_ok=True)

# Initialize embeddings
embeddings = OpenAIEmbeddings(
    base_url="https://openai.vocareum.com/v1",
    api_key=os.getenv("VOCAREUM_API_KEY")
)

# Create the vector store
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    persist_directory=persist_directory
)

print(f"Vector store created and persisted to {persist_directory}")
print(f"Total vectors stored: {vectorstore._collection.count()}")

Vector store created and persisted to data/vectorstore
Total vectors stored: 27


## 5. Test the RAG Pipeline


In [6]:
# Test the search functionality
# Try different queries related to energy optimization
# Test queries like:
# - "electric vehicle charging tips"
# - "thermostat optimization"
# - "dishwasher energy saving"
# - "solar power maximization"

test_queries = [
    "electric vehicle charging tips",
    "thermostat optimization",
    "dishwasher energy saving",
    "solar power maximization",
    "HVAC system efficiency",
    "pool pump scheduling",
    "pre-cooling before peak prices",
    "home battery time-of-use arbitrage",
    "winter energy saving",
    "smart plug standby power",
]

print("=== Testing Vector Search ===")
for query in test_queries:
    print(f"\nQuery: '{query}'")
    docs = vectorstore.similarity_search(query, k=2)
    for i, doc in enumerate(docs):
        print(f"  Result {i+1} [{os.path.basename(doc.metadata['source'])}]: {doc.page_content[:100]}...")

=== Testing Vector Search ===

Query: 'electric vehicle charging tips'


  Result 1 [tip_ev_charging.txt]: Choosing the Best Time to Charge:
- Avoid charging during peak hours (commonly 4 PM to 9 PM on weekd...
  Result 2 [tip_ev_charging.txt]: Electric Vehicle Charging Optimization

Home EV charging is often the largest single electricity loa...

Query: 'thermostat optimization'


  Result 1 [tip_hvac_optimization.txt]: HVAC Optimization Strategies

Heating and cooling are usually the largest share of a home's energy u...
  Result 2 [tip_energy_savings.txt]: Use power strips to easily turn off multiple devices at once. Many electronics continue to draw powe...

Query: 'dishwasher energy saving'


  Result 1 [tip_device_best_practices.txt]: Dishwasher Best Practices:
- Only run when completely full
- Use the energy-saving or eco mode when ...
  Result 2 [tip_smart_home_automation.txt]: Price- and Solar-Aware Scheduling:
- Connect flexible loads such as EV chargers, dishwashers, washin...

Query: 'solar power maximization'


  Result 1 [tip_renewable_energy_integration.txt]: Renewable Energy Integration

Rooftop solar produces the most electricity in the middle of the day, ...
  Result 2 [tip_renewable_energy_integration.txt]: Using Weather Forecasts:
- Plan solar-heavy tasks for sunny or partly cloudy days. On cloudy or rain...

Query: 'HVAC system efficiency'


...

  Result 1 [tip_smart_home_automation.txt]: Price- and Solar-Aware Scheduling:
- Connect flexible loads such as EV chargers, dishwashers, washin...
  Result 2 [tip_device_best_practices.txt]: Dishwasher Best Practices:
- Only run when completely full
- Use the energy-saving or eco mode when ...

Query: 'pre-cooling before peak prices'


  Result 1 [tip_hvac_optimization.txt]: Pre-Cooling and Pre-Heating Around Peak Prices:
- When time-of-use prices spike in the afternoon (co...
  Result 2 [tip_time_of_use_pricing.txt]: Shifting Loads to Save Money:
- Flexible loads are devices whose timing does not matter much: EV cha...

Query: 'home battery time-of-use arbitrage'


  Result 1 [tip_energy_storage_optimization.txt]: Operating Modes:
- Self-consumption mode: charge from excess solar during the day and discharge in t...
  Result 2 [tip_time_of_use_pricing.txt]: Shifting Loads to Save Money:
- Flexible loads are devices whose timing does not matter much: EV cha...

Query: 'winter energy saving'


  Result 1 [tip_energy_savings.txt]: Saving energy at home can be simple and effective. Turn off lights when not in use and unplug device...
  Result 2 [tip_seasonal_energy_management.txt]: Fall:
- Schedule an HVAC tune-up before heating season and replace the air filter.
- Seal drafts aro...

Query: 'smart plug standby power'


  Result 1 [tip_smart_home_automation.txt]: Smart Plugs and Power Strips:
- Standby or phantom loads from TVs, game consoles, chargers and compu...
  Result 2 [tip_energy_savings.txt]: Use power strips to easily turn off multiple devices at once. Many electronics continue to draw powe...


## 6. Test the Search Tool


In [7]:
# Test the search_energy_tips tool from tools.py
# Import and test the tool with various queries
# Verify that it returns relevant results

from tools import search_energy_tips

# Test the search_energy_tips function
print("=== Testing search_energy_tips Tool ===")

test_queries = [
    "electric vehicle charging",
    "thermostat settings",
    "dishwasher optimization",
    "solar power tips",
    "battery storage strategy",
    "seasonal thermostat settings",
]

for query in test_queries:
    print(f"\nQuery: '{query}'")
    result = search_energy_tips.invoke(
        input={
            "query": query, 
            "max_results": 3,
        }
    )
    
    if "error" in result:
        print(f"  Error: {result['error']}")
    else:
        print(f"  Found {result['total_results']} results")
        for i, tip in enumerate(result['tips']):
            print(f"    {i+1}. {tip['content'][:100]}...")
            print(f"       Source: {tip['source']}")
            print(f"       Relevance: {tip['relevance_score']}")

=== Testing search_energy_tips Tool ===

Query: 'electric vehicle charging'


  Found 3 results
    1. Electric Vehicle Charging Optimization

Home EV charging is often the largest single electricity loa...
       Source: tip_ev_charging.txt
       Relevance: high
    2. Scheduling Strategies:
- On sunny days, let solar fill the battery before noon if possible, then use...
       Source: tip_energy_storage_optimization.txt
       Relevance: high
    3. Choosing the Best Time to Charge:
- Avoid charging during peak hours (commonly 4 PM to 9 PM on weekd...
       Source: tip_ev_charging.txt
       Relevance: medium

Query: 'thermostat settings'


  Found 3 results
    1. HVAC Optimization Strategies

Heating and cooling are usually the largest share of a home's energy u...
       Source: tip_hvac_optimization.txt
       Relevance: high
    2. Fall:
- Schedule an HVAC tune-up before heating season and replace the air filter.
- Seal drafts aro...
       Source: tip_seasonal_energy_management.txt
       Relevance: high
    3. Use power strips to easily turn off multiple devices at once. Many electronics continue to draw powe...
       Source: tip_energy_savings.txt
       Relevance: medium

Query: 'dishwasher optimization'


  Found 3 results
    1. Dishwasher Best Practices:
- Only run when completely full
- Use the energy-saving or eco mode when ...
       Source: tip_device_best_practices.txt
       Relevance: high
    2. Price- and Solar-Aware Scheduling:
- Connect flexible loads such as EV chargers, dishwashers, washin...
       Source: tip_smart_home_automation.txt
       Relevance: high
    3. Large devices like electric vehicles, washing machines and dishwashers often support delayed start o...
       Source: tip_device_best_practices.txt
       Relevance: medium

Query: 'solar power tips'


  Found 3 results
    1. Using Weather Forecasts:
- Plan solar-heavy tasks for sunny or partly cloudy days. On cloudy or rain...
       Source: tip_renewable_energy_integration.txt
       Relevance: high
    2. Net Metering and Export:
- Under net metering, excess solar is credited on your bill. Under newer ta...
       Source: tip_renewable_energy_integration.txt
       Relevance: high
    3. Seasonal Energy Management

Energy use and solar production change throughout the year. Adjusting ha...
       Source: tip_seasonal_energy_management.txt
       Relevance: medium

Query: 'battery storage strategy'


  Found 3 results
    1. Energy Storage Optimization

Home batteries store energy for use later, letting a household use sola...
       Source: tip_energy_storage_optimization.txt
       Relevance: high
    2. Scheduling Strategies:
- On sunny days, let solar fill the battery before noon if possible, then use...
       Source: tip_energy_storage_optimization.txt
       Relevance: high
    3. Water Heaters as Thermal Storage:
- A water heater can store energy as heat. Heat water during solar...
       Source: tip_energy_storage_optimization.txt
       Relevance: medium

Query: 'seasonal thermostat settings'


  Found 3 results
    1. HVAC Optimization Strategies

Heating and cooling are usually the largest share of a home's energy u...
       Source: tip_hvac_optimization.txt
       Relevance: high
    2. Fall:
- Schedule an HVAC tune-up before heating season and replace the air filter.
- Seal drafts aro...
       Source: tip_seasonal_energy_management.txt
       Relevance: high
    3. Seasonal Energy Management

Energy use and solar production change throughout the year. Adjusting ha...
       Source: tip_seasonal_energy_management.txt
       Relevance: medium
